In [ ]:
! pip install datasets==4.0.0 transformers==4.54.0 \
    smolagents==1.20.0 \
    langchain==0.3.27 langgraph==0.2.45 \
    langchain_together==0.3.0 together==1.5.21 \
    llama_index==0.12.52 matplotlib==3.10.3

# Домашнее задание по теме "Агенты"
Данное домашнее задание состоит из двух частей: дизайна агента для решения реальных задач (наш мини-конкурент openai/anthropic/perplexity) и обучения модели function calling.

# Часть 1. GAIA - 25 баллов

В данном задании вам необходимо спроектировать агента для ответа на открытые вопросы. Для этого вам необходимо выбрать модель и написать для нее необходимые инструменты.

Постарайтесь проверить себя в конце с помощью функции `llm_as_a_judge_eval` и написать агента, который отвечает на наибольшее число вопросов.
В данном задании будет оцениваться дизайн функций - достаточно ли покрыт агент инструментами для ответа на вопросы.

Вначале давайте загрузим вопросы, на которых мы будем строить нашего агента.

In [1]:
import json
from typing import List, Dict
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from datasets import load_dataset

In [2]:
dataset = load_dataset("gaia-benchmark/GAIA", "2023_level1")

Using the latest cached version of the dataset since gaia-benchmark/GAIA couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '2023_level1' at /Users/hawkeoni/.cache/huggingface/datasets/gaia-benchmark___gaia/2023_level1/0.0.1/ec492fe4320ee795b1aed6bb46229c5f693226b0f1316347501c24b4baeee005 (last modified on Sun Jul 20 20:39:00 2025).


In [3]:
task_ids = set(["8e867cd7-cff9-4e6c-867a-ff5ddc2550be",
 "a1e91b78-d3d8-4675-bb8d-62741b4b68a6",
 "2d83110e-a098-4ebb-9987-066c06fa42d0",
 "cca530fc-4052-43b2-b130-b30968d8aa44",
 "4fc2f1ae-8625-45b5-ab34-ad4433bc21f8",
 "6f37996b-2ac7-44b0-8e68-6d28256631b4",
 "9d191bce-651d-4746-be2d-7ef8ecadb9c2",
 "cabe07ed-9eca-40ea-8ead-410ef5e83f91",
 "3cef3a44-215e-4aed-8e3b-b1e3f08063b7",
 "99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3",
 "305ac316-eef6-4446-960a-92d80d542f82",
 "f918266a-b3e0-4914-865d-4faa564f1aef",
 "3f57289b-8c60-48be-bd80-01f8099ca449",
 "1f975693-876d-457b-a649-393859e79bf3",
 "840bfca7-4f7b-481a-8794-c560c340185d",
 "bda648d7-d618-4883-88f4-3466eabd860e",
 "cf106601-ab4f-4af9-b045-5295fe67b37d",
 "a0c07678-e491-4bbc-8f0b-07405144218f",
 "7bd855d8-463d-4ed5-93ca-5fe35145f733",
 "5a0c1adf-205e-4841-a666-7c3ef95def9d"
])
assert len(task_ids) == 20
samples = [sample for sample in dataset["validation"] if sample["task_id"] in task_ids]
assert len(samples) == len(task_ids), "Что-то не так с заданием, обратитесь, пожалуйста, в чат!"

In [ ]:
print(json.dumps(samples[0], indent=4))

# Альтернативная загрузка датасета
Датасет поломался https://huggingface.co/datasets/gaia-benchmark/GAIA/discussions/39
Скачайте архив из сайта учебной платформы и загрузите его функцией `load_dataset_from_folder`

In [14]:
import os
import json
from pathlib import Path

def load_dataset_from_folder(input_dir):
    dataset = []
    files_dir = os.path.join(input_dir, "files")
    
    for fname in os.listdir(input_dir):
        if not fname.endswith(".json"):
            continue
        
        with open(os.path.join(input_dir, fname), "r", encoding="utf-8") as f:
            item = json.load(f)

        if item["file_name"]:
            # Construct corrected path
            corrected_path = os.path.join(files_dir, item["file_name"])
            item["file_path"] = str(Path(corrected_path).absolute()) if os.path.exists(corrected_path) else ""
        
        dataset.append(item)
    
    return dataset

# Example usage
samples = load_dataset_from_folder("gaia_dataset")
assert len(samples) == 20
for sample in samples:
    if sample["file_path"]:
        assert Path(sample["file_path"]).exists()



"""
# Код сериализации датасета, его можно не запускать
import os
import json
import shutil
from uuid import uuid4

def serialize_dataset(data, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files_dir = os.path.join(output_dir, "files")
    os.makedirs(files_dir, exist_ok=True)

    for item in data:
        task_id = item["task_id"]
        file_path = item.get("file_path")
        
        # Copy file if it exists
        if file_path and os.path.isfile(file_path):
            ext = os.path.splitext(file_path)[1]
            new_filename = f"{task_id}{ext}"
            new_file_path = os.path.join(files_dir, new_filename)
            shutil.copy(file_path, new_file_path)
            item["file_path"] = new_file_path
        else:
            item["file_path"] = ""

        # Save the JSON item
        with open(os.path.join(output_dir, f"{task_id}.json"), "w", encoding="utf-8") as f:
            json.dump(item, f, indent=2)


# serialize_dataset(samples, "gaia_dataset")
"""
print("ALL GOOD")

ALL GOOD


В данном датасете нас интересуют следующие поля:
* Question - это запрос, который приходит в нашу агентскую систему
* file_path - путь к файлу, по которому задается вопрос. 5 примеров из датасета требуют работы с файлами, необходимо изучить их и написать соответствующие функции. **Не забудьте подать путь к файлу вместе с вопросом!**
* Annotator Metadata - словарь, в котором описаны инструменты и действия человека при решении этой задачи. Вы можете изучить аннотации, чтобы понять, какие инструменты вам необходимо написать.
* Final answer - ответ на задачу. Не подавайте его в модель, мы будем использовать это поле для проверки ответа

Пример задачи с файлом **(файл не обязательно является картинкой)**:

In [ ]:
print(json.dumps(samples[3], indent=4))
image_path = samples[3]["file_path"]
img = mpimg.imread(image_path)
plt.imshow(img)

In [ ]:
import os
from smolagents import ChatMessage, OpenAIServerModel

model = OpenAIServerModel(
    model_id="Qwen/Qwen3-235B-A22B-fp8-tput",
    api_base="https://api.together.xyz/v1/",
    api_key=os.environ["TOGETHER_API_KEY"],
)

In [ ]:
response = model.generate(messages=[ChatMessage(role="user", content="Hello!")])
print(response)
print(response.content)

Функция для проверки ответов

In [ ]:
def llm_as_a_judge_eval(dataset: List[Dict[str, str]], answers: List[str]):
    model_id = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
    prompt = (
        "Your task is to compare the answers to the question {question}. "
        "Reference answer: `{ref}` Answer to Compare: `{ans}` "
        "Generate only one word `TRUE` if the answers match and `FALSE` if the answer is incorrect"
        "If you are unsure or the format is wrong then the answer is wrong"
    )
    model = OpenAIServerModel(
        model_id=model_id,
        api_base="https://api.together.xyz/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
    )
    assert len(dataset) == len(answers)
    correct = 0
    for sample, answer in zip(dataset, answers):
        sample_prompt = prompt.format(question=sample["Question"], ref=sample["Final answer"], ans=answer)
        response = model.generate(messages=[ChatMessage(role="user", content=sample_prompt)]).content
        if "TRUE" in response:
            correct += 1
        elif "FALSE" in response:
            continue
        else:
            raise ValueError(f"LLM-as-a-judge failed, response is: {response}")
    return correct
        

Давайте посмотрим, как справится агент с инструментом "Поиск" с данной задачей на одном вопросе

In [ ]:
from smolagents import CodeAgent, WebSearchTool, ToolCallingAgent

# Если поиск не работает, можно воспользоваться https://app.tavily.com/home
# интеграция с ним есть в langchain https://python.langchain.com/api_reference/community/tools/langchain_community.tools.tavily_search.tool.TavilySearchResults.html
agent = ToolCallingAgent(tools=[WebSearchTool()], model=model, stream_outputs=False)

r = agent.run(
    samples[0]["Question"],
    max_steps=3
)

In [ ]:
print(r)

Так как модель может выдавать очень длинные ответы, давайте напишем функцию, которая достает из модели короткую версию ответа

In [ ]:

def summarize_answer(question: str, model_response: str) -> str:
    model_id = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
    # ---- Ваш код здесь ----
    prompt = ""
    response = ""
    # Вам необходимо написать промпт для получения короткого ответа
    # и получить выходы модели
    # ---- Конец кода ----
    
    return response

In [ ]:
short_answer = summarize_answer(
    "How many odd numbers are there in [1, 2, 3, 4, 5]", 
    "There are 3 odd numbers, they are [1, 3, 5]"
)
print(short_answer)
# Если на вашем промпте ассерт не проходит, но ответ короткий и подходит по смыслу,
# то игнорируйте следующую проверку
assert ("3" in short_answer or "three" in short_answer.lower()) and len(short_answer) < 10

In [ ]:
correct = llm_as_a_judge_eval(samples[:1], samples[0]["Final answer"])
assert correct == 1

Вы можете написать агента на любом удобном для вас фреймворке - swarm, langchain, llama_index, smolagents, самописный агент. Вы можете использовать любую модель и любые функции. Функции можно описывать самим или брать готовые, даже подключаться ко внешним MCP.

Для удобства для вас предоставлены несколько шаблонов агентов, вы можете как расширить их, так и писать с нуля.
Вы также не ограничены версиями установленными версиями библиотек, их можно менять.

Рекомендую обратить внимание на библиотеку langchain_community, в ней есть множество полезных [инструментов](https://python.langchain.com/api_reference/community/tools.html), таких как:
* [wikipedia search](https://python.langchain.com/docs/integrations/document_loaders/wikipedia/)
* [поиск tavily](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.tavily_search.tool.TavilySearchResults.html)

Даже если вы не используете langchain, вы можете обернуть эти тулзы в функции и классы для вашего фреймворка.

Для описания картинок можно воспользоваться моделью [qwen 2.5 vl](https://www.together.ai/models/qwen2-5-vl-72b-instruct) - использовать ее как готовый инструмент для описания картинок с фиксированным промптом.

## SmolAgents

In [ ]:
# можно использовать CodeAgent
agent = ToolCallingAgent(tools=[WebSearchTool()], model=model, stream_outputs=False)

## LangChain + LangGraph

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_together import ChatTogether

llm = ChatTogether(model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
                   api_key=os.getenv("TOGETHER_API_KEY"))


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_react_agent(
    model=llm,
    tools=[get_weather],
)

# Run the agent
r = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)
print(r["messages"][-1].content)

## Llama Index

In [ ]:
from llama_index.llms.together import TogetherLLM
from llama_index.core.llms import ChatMessage
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent


llm = TogetherLLM(model="meta-llama/Llama-3.3-70B-Instruct-Turbo", api_key=os.environ["TOGETHER_API_KEY"])
weather_tool = FunctionTool.from_defaults(fn=get_weather)
agent = ReActAgent(tools=[weather_tool], llm=llm, memory=None, verbose=True)
r = agent.chat("what is the weather in sf")

Ваша реализация может быть на любом удобном фреймворке. Вы можете как проверить ответы сами, так и послать список с ответами для ручной проверки.

In [ ]:
# ---- Ваш код здесь ----
MyAwesomeAgent = ...
# ---- Конец кода ----



# Часть 2. Обучение модели - 25 баллов
В данном задании мы возьмем модель `HuggingFaceTB/SmolLM-360M-Instruct` и обучим ее Function Calling. Базовая модель не умеет пользоваться инструментами, мы добавим в нее эту возможность с помощью Finetuning'а.

В реальных сценариях вам придется брать более умную instruct-модель, которая уже умеет в Function Calling, и обучать ее на логах моделей по типу chatgpt-o*, anthropic claude, deepseek-r1.

Принцип дообучения и переноса знаний сохраняется - необходимо получить корпус вызова инструментов (написать самим, сгенерировать синтетически, получить от более умной модели), а потом дообучить на нем модель. Для наглядности процесса в данном задании специально выбрана модель, которая ни разу во время обучения не видела сценария вызова функций.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-360M-Instruct").to(device)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-360M-Instruct")


In [ ]:
print(tokenizer.chat_template)

Как видно из `tokenizer.chat_template` наша модель даже не имеет формата вызова функций и наивно пропишет роль tool при подстановке диалога с ответом инструмента.

In [ ]:
messages = [{"role": "user", "content": "Hello there"}, {"role": "tool", "content": "test_output"}]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

Давайте обновим шаблон нашего токенизатора, чтобы ответ от инструмента имел дополнительные тэги

In [ ]:
tokenizer.chat_template = """
{% for message in messages %}
{% if message['role'] == 'tool' %}
<|im_start|>tool
<tool_response>
{{ message['content'] }}
</tool_response><|im_end|>
{% else %}
{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
{{ '<|im_start|>assistant\n' }}
{% endif %}
""".strip()

In [ ]:
messages = [{"role": "user", "content": "Hello there"}, {"role": "tool", "content": "test_output"}]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

Загрузим датасет, на котором будем обучать нашу модель вызывать функции.

In [ ]:
dataset = load_dataset("Jofthomas/hermes-function-calling-thinking-V1")

In [ ]:
print(
    tokenizer.apply_chat_template(
        dataset["train"][0]["conversations"][:3], 
        add_generation_prompt=False, tokenize=False)
)

Данный датасет состоит из диалогов с использованием инструментов, формат вызова инструментов уже описан в первом сообщении с ролью system. Как правило формат подачи функций в модель уже содержится в `chat_template`, но в данном задании для простоты мы будем пользоваться готовым промптом.

Давайте посмотрим, как справится наша модель без дообучения.

In [ ]:
text = tokenizer.apply_chat_template(
    dataset["train"][0]["conversations"][:2], add_generation_prompt=True, tokenize=False
)
inputs = tokenizer(text, return_tensors="pt").to(device)

In [ ]:
res = model.generate(**inputs)
print(tokenizer.decode(res[0, inputs.input_ids.size(1):].tolist()))

Как видно - модель не генерирует вызов инструмента несмотря на то, что эта логика описана в промпте.
Настало время дообучить модель!

Первым делом мы займемся обработкой входных данных:
1. Оставим только диалоги длины <= 5
2. При генерации tool calls в данном корпусе происходит ризонинг в тэгах `<think>...</think>`. Мы удалим этот текст, т.к. нашей модели будет сложно выучить ризонинг и вызов инструментов одновременно с нуля.
3. Переименуем роли human->user, model->assistant
4. Уберем из диалогов `<tool_response>` и `</tool_response>`, так как мы уже добавили это форматирование в наш `tokenizer.chat_template`

In [ ]:
import re


filtered_dataset = []
role_mapping = {"human": "user", "model": "assistant", "system": "system", "tool": "tool"}
for sample in dataset["train"]:
    # ---- Ваш код здесь ----
    # 1. Отфильтровать по длине
    # 2. Убрать <think>...</think>
    # 3. Поменять роли
    # 4. Убрать из сообщений <tool_response> и </tool_response>
    # ---- Конец кода ----
    
assert len(filtered_dataset) == 505
reference = """\n{\'joke\': "Why don\'t scientists trust atoms? Because they make up everything!"}\n"""
assert (filtered_dataset[0]["conversations"][3]["content"] == reference)

In [ ]:
print(tokenizer.apply_chat_template(filtered_dataset[0]["conversations"], tokenize=False))

In [ ]:
filtered_dataset[0]["conversations"][1]

Теперь допишем функции для обучения модели. В данном сценарии нам всегда нужно учиться на нескольких фразах
от модели, типичный диалог выглядит следующим образом:

1. System prompt
2. Запрос от юзера
3. **Генерация tool call**
4. Ответ инструмента
5. **Ответ от модели с учетом ответа инструмента**

Нам необходимо учиться на шагах диалога, выделенных жирным текстом. Для этого давайте напишем функцию 
`get_assistant_message_positions`, которая будет возвращать список кортежей - индексы токенов начала и конца
ответа модели.


In [ ]:
from typing import List, Tuple, Dict

def get_assistant_message_positions(messages: Dict[str, str]) -> List[Tuple[int, int]]:
    # ---- Ваш код здесь ----
    positions = []
#     for idx, message in enumerate(messages):
#         if message["role"] == "assistant":
#             # Не забудьте корректно проставить флаг add_generation_prompt
#             pass
    
#     return positions
    
    # ---- Конец кода ----
    

assert get_assistant_message_positions(filtered_dataset[0]["conversations"]) == [(388, 418), (460, 495)]
assert get_assistant_message_positions(filtered_dataset[1]["conversations"]) == [(388, 432), (493, 537)]


Дальше давайте напишем функцию `collate_fn`:
* Для каждого сэмпла получаем индексы токенов с ролью assistant
* Токенизируем текст, получаем словарь с input_ids, attention_mask ...
* Добавляем в словарь метки под названием labels, **учим только токены ассистента**

In [ ]:
@torch.no_grad()
def collate_fn(examples: List[Dict]):
    # ---- Ваш код здесь ----
#     assistant_token_positions = [
#         get_assistant_message_positions(example["conversations"]) for example in examples
#     ]
#     texts = [
#         tokenizer.apply_chat_template(ex["conversations"], tokenize=False, add_generation_prompt=False) 
#         for ex in examples
#     ]
    batch = tokenizer(...)
    ...
    return batch

    # ---- Конец кода ----



In [ ]:
batch = collate_fn([filtered_dataset[0], filtered_dataset[1]])

assert tokenizer.decode(batch["labels"][0][batch["labels"][0] != -100]).strip() == """
<tool_call>
{'name': 'get_random_joke', 'arguments': {}}
</tool_call><|im_end|>

<|im_start|>Sure, here's a joke for you: "Why don't scientists trust atoms? Because they make up everything!" I hope that brings a smile to your face.<|im_end|>

<|im_end|>
""".strip()

assert tokenizer.decode(batch["labels"][1][batch["labels"][1] != -100]).strip() == """
<tool_call>
{'name': 'get_stock_prices', 'arguments': {'companies': ['Apple', 'Microsoft', 'Tesla']}}
</tool_call><|im_end|>

<|im_start|>The current stock prices are as follows: Apple is at $150.75, Microsoft is at $305.22, and Tesla is at $780.79.<|im_end|>
""".strip()

Далее давайте используем Transformers Trainer для обучения модели на наших данных. Предлагается пройти одну эпоху с батчом 4 (используйте аккумуляцию градиентов и чекпоинтинг градиентов/активаций) и lr = 1e-4

In [ ]:
from transformers import TrainingArguments, Trainer

# ---- Ваш код здесь ----
# ---- Конец кода ----


Теперь давайте проверим, сможет ли наша модель вызвать функцию на примере, который она не видела во время обучения.

In [ ]:
for sample in dataset["train"]:
    if len(sample["conversations"]) > 5:
        break
assert len(sample["conversations"]) > 5

text = tokenizer.apply_chat_template(
    sample["conversations"][:2], add_generation_prompt=True, tokenize=False
)
inputs = tokenizer(text, return_tensors="pt").to(device)
res = model.generate(**inputs, use_cache=False)
print(tokenizer.decode(res[0, inputs.input_ids.size(1):].tolist()))

Если все прошло хорошо, то модель должна выдать что-то похожее на
```
<tool_call>
{'name': 'get_stock_price', 'arguments': {'company': 'Apple'}}
</tool_call><|im_end|>
```